# Hypothesis Test: Does Sentiment Differ Between iOS and Android?

Beli's overall store ratings differ by platform: 4.9 on the App Store versus 4.2 on Google Play. Within my sample (users who left a written review), the gap persists but both averages are lower: 3.41 on the App Store and 3.11 on Google Play. I want to test whether this difference in sampled ratings is statistically significant or attributable to chance.

**Question:** Do Beli users on the App Store (iOS) and Google Play (Android) give different overall star ratings?

**Method:** A permutation test on star ratings.
- **H0 (null):** the two platforms have the same mean rating — any observed gap is sampling noise.
- **H1 (alternative):** the two platforms have different mean ratings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42) 

In [ ]:
df = pd.read_csv("data/tagged_reviews.csv")
df = df[["source", "rating"]].dropna()
df.head()

In [ ]:
app_store_mean_rating = df[df['source'] == 'app_store']['rating'].mean()
app_store_mean_rating

In [ ]:
google_play_mean_rating = df[df['source'] == 'google_play']['rating'].mean()
google_play_mean_rating

### The test statistic

I compare the **difference in mean rating** between the two platforms. This helper
computes that difference for any grouping label — I'll reuse it for both the real
data and the shuffled data, so the calculation is identical every time.

In [ ]:
def difference_of_means(data, value_col, group_col):
    """Difference in mean of `value_col` between the two groups in `group_col`.
    Returns: group[1] mean - group[0] mean (alphabetical order)."""
    means = data.groupby(group_col)[value_col].mean()
    return means.iloc[1] - means.iloc[0]

In [ ]:
observed_diff = difference_of_means(df, "rating", "source")
print(f"Observed difference in mean rating: {observed_diff:.3f}")

### One shuffle

If platform truly doesn't matter (H0), the labels are arbitrary — I can shuffle them
and the difference should be near zero on average. This function shuffles the platform
labels once and recomputes the difference. Running it many times builds the
**null distribution**: what the difference looks like when platform is random.

In [ ]:
def one_simulated_difference(data, value_col, group_col):
    """Shuffle the group labels once, then recompute the difference of means."""
    shuffled = data[[value_col]].copy()
    shuffled["shuffled_label"] = rng.permutation(data[group_col].values)
    return difference_of_means(shuffled, value_col, "shuffled_label")

### Build the null distribution

Repeat the shuffle 10,000 times, collecting the difference each time. This is the
distribution of differences we'd expect if the two platforms were truly the same.

In [ ]:
n_iter = 10000
simulated_diffs = np.array([
    one_simulated_difference(df, "rating", "source")
    for _ in range(n_iter)
])

### Compare observed vs. null

The histogram is the null distribution; the red line is the difference we actually
observed. If the red line sits in the bulk, the platforms don't meaningfully differ.
If it sits far in a tail, the difference is unlikely to be chance.

In [ ]:
plt.hist(simulated_diffs, bins=20, edgecolor="white")
plt.axvline(observed_diff, color="red", lw=2, label="observed")
plt.title("Difference in Mean Rating Under H0 (platform shuffled)")
plt.xlabel("google_play mean − app_store mean")
plt.ylabel("frequency")
plt.legend()
plt.show()

### P-value

The two-sided p-value is the fraction of shuffled differences at least as extreme
(in either direction) as the observed one. Small p (< 0.05) means a gap this large
is unlikely under H0, so we reject it.

In [ ]:
p_value = np.mean(np.abs(simulated_diffs) >= np.abs(observed_diff))
print(f"Observed difference: {observed_diff:.3f}")
print(f"Two-sided p-value:   {p_value:.4f}")

## Conclusion

Two-sided permutation test: **p = 0.0153**. Since the p-value is below 0.05, we reject the null: iOS and Android users rate Beli differently on average.

The observed difference was **−0.306 stars** (google_play − app_store). iOS users rated Beli about 0.31 stars higher than Android users. The result is statistically significant but the effect is small (0.31 on a 5-point scale), so I'm confident the gap is not chance, not that it's large in practical terms.

**Interpretation & limitations:**
- Platform was self-selected, not randomly assigned. Users choose iOS vs Android for reasons correlated with demographics and spending, so this shows an association between platform and rating — not that the platform causes different satisfaction. This is why it's a significance test, not an A/B experiment.
- Selection bias: reviewers self-select toward extremes, so this reflects vocal users, not the median user.